In [13]:
import torch
import random

# ==========================================
# 第一步：生成人造数据集 (假数据)
# ==========================================
def synthetic_data(w, b, num_examples):
    """生成 y = Xw + b + 噪声"""
    # TODO 1: 使用 torch.normal 生成均值为0，方差为1的随机特征 X。
    # 提示: X 的形状应该是 (num_examples, len(w))
    X = torch.normal(0,1,(num_examples,len(w)))
    
    # TODO 2: 计算真实的 y (提示: 矩阵乘法可以用 torch.matmul(X, w) 或 X @ w)
    y = X@w + b
    
    # TODO 3: 给 y 加上一点均值为0，方差为0.01的随机噪声 (使用 torch.normal)
    # 提示: 噪声的形状要和 y 一样
    y += torch.normal(0,0.01,y.shape)
    
    return X, y.reshape((-1, 1))

# 定义真实的权重和偏差（作为评判标准）
true_w = torch.tensor([2.0, -3.4])
true_b = 4.2
# 调用你写的函数生成 1000 个样本
features, labels = synthetic_data(true_w, true_b, 1000)
print("features shape:", features.shape) # 期待看到[1000, 2]
print("labels shape:", labels.shape)     # 期待看到[1000, 1]


# ==========================================
# 第二步：初始化模型参数 (炼丹炉的初始状态)
# ==========================================
# TODO 4: 初始化我们要训练的权重 w 和偏差 b。
# 要求: w 初始化为均值为0、方差为0.01的正态分布，形状为 (2, 1)。
#       b 初始化为全 0，形状为 1。
# 致命提示: 这两个张量都需要计算梯度！必须加上 requires_grad=True
w = torch.normal(0,0.01,(2,1),requires_grad=True)
b = torch.zeros(1, requires_grad=True)


# ==========================================
# 第三步：定义核心组件 (模型、损失、优化器)
# ==========================================
def linreg(X, w, b):
    """线性回归模型"""
    # TODO 5: 返回线性回归的预测值 (X 和 w 的矩阵乘法，再加上 b)
    return X@w + b

def squared_loss(y_hat, y):
    """均方损失 (MSE)"""
    # TODO 6: 返回 (预测值 - 真实值) 的平方，再除以 2。
    # 提示: 返回的形状应该和 y_hat 一样，不要在这里求均值，一会在外部求。
    return (y-y_hat)**2/2

def sgd(params, lr, batch_size):
    """小批量随机梯度下降"""
    # 提示: 更新参数时，不应该记录这些操作的梯度，所以要用 torch.no_grad()
    with torch.no_grad():
        for param in params:
            # TODO 7: 将参数减去 学习率(lr) * 梯度(param.grad) / batch_size
            param -= lr * param.grad/batch_size
            # TODO 8: 梯度清零！(这步极其重要，否则梯度会累加：param.grad.zero_())
            param.grad.zero_()


# ==========================================
# 第四步：写训练循环 (炼丹开始！)
# ==========================================
# 这里为了简化，我帮你写一个极简的数据迭代器（每次抛出 batch_size 个数据）
def data_iter(batch_size, features, labels):
    num_examples = len(features)
    indices = list(range(num_examples))
    random.shuffle(indices) # 打乱顺序
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(indices[i: min(i + batch_size, num_examples)])
        yield features[batch_indices], labels[batch_indices]

# 超参数设置
lr = 0.03
num_epochs = 3
batch_size = 10
net = linreg
loss = squared_loss

for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        # 1. 前向传播：把 X 扔进网络，算出预测值
        l = loss(net(X, w, b), y)
        
        # 2. 反向传播：计算 l 对参数(w,b)的梯度
        # 提示: l 的形状是 [10, 1]，我们需要把它求和变成标量，再调用 .backward()
        # TODO 9: l.sum().backward()
        l.sum().backward()
        
        # 3. 更新参数：调用你写的 sgd 函数更新 w 和 b
        # TODO 10: sgd([w, b], lr, batch_size)
        sgd([w,b], lr, batch_size)
        
    # 每个 epoch 结束，看看当前训练的有多好
    with torch.no_grad():
        train_l = loss(net(features, w, b), labels)
        print(f'epoch {epoch + 1}, loss {float(train_l.mean()):f}')

# ==========================================
# 验收结果！
# ==========================================
print(f'w 的估计误差: {true_w - w.reshape(true_w.shape)}')
print(f'b 的估计误差: {true_b - b}')

features shape: torch.Size([1000, 2])
labels shape: torch.Size([1000, 1])
epoch 1, loss 0.039975
epoch 2, loss 0.000158
epoch 3, loss 0.000053
w 的估计误差: tensor([ 0.0005, -0.0003], grad_fn=<SubBackward0>)
b 的估计误差: tensor([0.0005], grad_fn=<RsubBackward1>)


In [8]:
import torch
import random

# ==========================================
# 第一步：生成人造数据集 (假数据)
# ==========================================
def synthetic_data(w, b, num_examples):
    """生成 y = Xw + b + 噪声"""
    # TODO 1: 使用 torch.normal 生成均值为0，方差为1的随机特征 X。
    # 提示: X 的形状应该是 (num_examples, len(w))
    X = torch.normal(0,1,(num_examples,len(w)))
    
    # TODO 2: 计算真实的 y (提示: 矩阵乘法可以用 torch.matmul(X, w) 或 X @ w)
    y = torch.mv(X, w) + torch.tensor(b)
    
    # TODO 3: 给 y 加上一点均值为0，方差为0.01的随机噪声 (使用 torch.normal)
    # 提示: 噪声的形状要和 y 一样
    y += torch.normal(0,0.01,y.shape)
    
    return X, y.reshape((-1, 1))

# 定义真实的权重和偏差（作为评判标准）
true_w = torch.tensor([2.0, -3.4])
true_b = 4.2
# 调用你写的函数生成 1000 个样本
features, labels = synthetic_data(true_w, true_b, 1000)
print("features shape:", features.shape) # 期待看到[1000, 2]
print("labels shape:", labels.shape)     # 期待看到[1000, 1]

features shape: torch.Size([1000, 2])
labels shape: torch.Size([1000, 1])
